<a href="https://colab.research.google.com/github/Mandira124/RAG-Based-Research-Assistant/blob/main/RAG_based_Research_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [59]:
#Install dependencies
!pip install -q sentence-transformers faiss-cpu groq PyPDF2 requests beautifulsoup4 wikipedia-api

In [60]:
# Imports and config
import os, pickle, re, requests
import numpy as np
import faiss
import PyPDF2
import wikipediaapi
from pathlib import Path
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import userdata, files

# ── Config ────────────────────────────────────────────
EMBED_MODEL   = "all-MiniLM-L6-v2"
GROQ_MODEL = "llama-3.1-8b-instant"
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 150
TOP_K         = 6
INDEX_DIR     = "index"
DOCS_DIR      = "data/docs"

os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(DOCS_DIR,  exist_ok=True)

# ── Clients ───────────────────────────────────────────
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Loading embedding model…")
embedder = SentenceTransformer(EMBED_MODEL)
print("Ready.")

Loading embedding model…


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready.


In [61]:
# Document loaders
def load_pdf(path):
    reader = PyPDF2.PdfReader(path)
    pages  = [p.extract_text() for p in reader.pages if p.extract_text()]
    return "\n".join(pages)

def load_txt(path):
    return Path(path).read_text(encoding="utf-8")

def load_url(url):
    resp = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(resp.text, "html.parser")
    # Remove nav, footer, script, style noise
    for tag in soup(["script","style","nav","footer","header","aside"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    # Collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def load_wikipedia(topic):
    wiki = wikipediaapi.Wikipedia(
        language="en",
        user_agent="ResearchAssistant/1.0"
    )
    page = wiki.page(topic)
    if not page.exists():
        raise ValueError(f"Wikipedia page not found: '{topic}'")
    return page.text

def load_source(source: str) -> tuple[str, str]:
    """
    Auto-detects source type.
    Returns (label, text) — label is used to tag chunks later.
    """
    if source.startswith("http://") or source.startswith("https://"):
        print(f"  Scraping URL: {source}")
        return source, load_url(source)
    elif source.startswith("wiki:"):
        topic = source[5:].strip()
        print(f"  Fetching Wikipedia: {topic}")
        return f"Wikipedia:{topic}", load_wikipedia(topic)
    elif source.endswith(".pdf"):
        print(f"  Reading PDF: {source}")
        return source, load_pdf(source)
    else:
        print(f"  Reading text file: {source}")
        return source, load_txt(source)

In [62]:
# Chunker
def chunk_text(text: str, label: str) -> list[dict]:
    """
    Returns list of dicts: {text, source, chunk_id}
    Keeping source metadata means the answer can cite where it came from.
    """
    chunks = []
    start  = 0
    cid    = 0
    while start < len(text):
        piece = text[start : start + CHUNK_SIZE].strip()
        # Skip garbage: chunks that are too short or mostly non-alphabetic
        alpha_ratio = sum(c.isalpha() for c in piece) / max(len(piece), 1)
        if len(piece) > 80 and alpha_ratio > 0.4:
            chunks.append({
                "text":     piece,
                "source":   label,
                "chunk_id": cid,
            })
            cid += 1
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks

In [63]:
import shutil

# Wipe old docs and index
shutil.rmtree("data/docs", ignore_errors=True)
shutil.rmtree("index",     ignore_errors=True)

# Recreate empty folders
os.makedirs("data/docs", exist_ok=True)
os.makedirs("index",     exist_ok=True)

# Clear conversation history too
conversation_history = []

print("Cleared. Now re-run Cell 5 to upload new documents.")

Cleared. Now re-run Cell 5 to upload new documents.


In [64]:
# ── Upload local files (PDF / TXT) ────────────────────
print("Upload your PDF or TXT files (cancel if you only have URLs/Wikipedia)")
try:
    uploaded = files.upload()
    for fname, content in uploaded.items():
        fpath = f"{DOCS_DIR}/{fname}"
        with open(fpath, "wb") as f:
            f.write(content)
        print(f"Saved: {fname}")
except Exception:
    print("No files uploaded, continuing.")

# ── Add URLs and Wikipedia topics ─────────────────────
# Edit this list freely. Format:
#   "https://..."       → scrape a webpage
#   "wiki:Topic Name"   → fetch Wikipedia article
extra_sources = [
    # "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
    # "wiki:Large language model",
    # "https://somesite.com/article",
]

# ── Build the index ───────────────────────────────────
all_chunks = []

# From uploaded files
for fname in os.listdir(DOCS_DIR):
    label, text = load_source(f"{DOCS_DIR}/{fname}")
    chunks      = chunk_text(text, label)
    all_chunks.extend(chunks)
    print(f"  → {len(chunks)} chunks")

# From URLs / Wikipedia
for src in extra_sources:
    label, text = load_source(src)
    chunks      = chunk_text(text, label)
    all_chunks.extend(chunks)
    print(f"  → {len(chunks)} chunks")

print(f"\nTotal chunks kept: {len(all_chunks)}")

# ── Embed ─────────────────────────────────────────────
print("Embedding… (takes ~30s for 500 chunks)")
texts      = [c["text"] for c in all_chunks]
embeddings = embedder.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings, dtype="float32")

# ── FAISS ─────────────────────────────────────────────
dim   = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

faiss.write_index(index, f"{INDEX_DIR}/faiss.index")
with open(f"{INDEX_DIR}/chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

print(f"Index built. Shape: {embeddings.shape}")

Upload your PDF or TXT files (cancel if you only have URLs/Wikipedia)


Saving DeepLearningMiniProject.pdf to DeepLearningMiniProject (3).pdf
Saved: DeepLearningMiniProject (3).pdf
  Reading PDF: data/docs/DeepLearningMiniProject (3).pdf
  → 25 chunks

Total chunks kept: 25
Embedding… (takes ~30s for 500 chunks)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Index built. Shape: (25, 384)


In [65]:
# Load index (run this if runtime restarts)
index = faiss.read_index(f"{INDEX_DIR}/faiss.index")
with open(f"{INDEX_DIR}/chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

print(f"Loaded {index.ntotal} vectors, {len(all_chunks)} chunks.")

Loaded 25 vectors, 25 chunks.


In [66]:
# Retriver
def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    q_vec = np.array(embedder.encode([query]), dtype="float32")
    distances, indices = index.search(q_vec, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx < len(all_chunks):
            chunk = all_chunks[idx].copy()
            chunk["score"] = float(dist)   # lower = more similar
            results.append(chunk)
    return results

In [67]:
conversation_history = []   # persists across ask() calls in the same session

def ask(question: str, top_k: int = TOP_K):
    global conversation_history

    # 1. Retrieve relevant chunks
    chunks = retrieve(question, top_k=top_k)
    context = "\n\n---\n\n".join(c["text"] for c in chunks)

    # 2. System prompt — sets the assistant's behaviour
    system_prompt = f"""You are a research assistant with access to a private knowledge base.
Answer questions using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information in the provided documents."

Important rules:
- Do NOT contradict or apologize for your previous answers unless the context clearly shows you were wrong.
- If a follow-up question is vague (like "elaborate"), expand on your previous answer using the context.
- Be specific — quote exact course names, methodologies, authors when present in context.
- Mention which document the information comes from.

Context:
{context}"""

    # 3. Build messages: system + full history + new user message
    messages = (
        [{"role": "system", "content": system_prompt}]
        + conversation_history
        + [{"role": "user", "content": question}]
    )

    # 4. Call Groq
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages,
        temperature=0.2,
        max_tokens=1024,
    )
    answer = response.choices[0].message.content

    # 5. Save this turn to history
    conversation_history.append({"role": "user",      "content": question})
    conversation_history.append({"role": "assistant", "content": answer})

    # 6. Print answer + sources
    print(f"\n{'='*60}")
    print(f"You: {question}")
    print(f"{'='*60}")
    print(f"\nAssistant:\n{answer}")
    print(f"\n{'─'*60}")
    print("Sources used:")
    for i, c in enumerate(chunks, 1):
        preview = c['text'][:120].replace('\n', ' ')
        print(f"  [{i}] {c['source']}  (score: {c['score']:.2f})")
        print(f"      \"{preview}…\"")

def reset_conversation():
    global conversation_history
    conversation_history = []
    print("Conversation history cleared.")

In [68]:
from IPython.display import display, clear_output
import ipywidgets as widgets

# ── UI elements ───────────────────────────────────────
text_input  = widgets.Text(
    placeholder="Ask a question about your documents…",
    layout=widgets.Layout(width="75%")
)
send_btn    = widgets.Button(description="Ask", button_style="primary")
reset_btn   = widgets.Button(description="Reset chat", button_style="warning")
output_area = widgets.Output()

def on_ask(b):
    question = text_input.value.strip()
    if not question:
        return
    text_input.value = ""          # clear input box
    with output_area:
        ask(question)

def on_reset(b):
    reset_conversation()
    with output_area:
        clear_output()
        print("Chat cleared. Ask a new question.")

send_btn.on_click(on_ask)
reset_btn.on_click(on_reset)

# Allow pressing Enter to submit
def on_submit(sender):
    on_ask(None)
text_input.on_submit(on_submit)

# ── Display ───────────────────────────────────────────
display(widgets.HBox([text_input, send_btn, reset_btn]))
display(output_area)

Output()